In [ ]:
import numpy as np
import pandas as pd
import io

csv_data = """xA,xB,xC,xD,xE,G
false,true,false,false,true,false
false,false,false,true,true,false
true,false,true,false,true,true
false,false,true,false,true,true
true,false,false,false,false,false
true,false,true,true,true,true
true,false,true,false,false,false
false,true,false,true,false,false
false,false,true,false,true,true
false,true,false,true,false,false
true,true,true,false,false,true
true,true,false,true,true,false
false,true,false,false,true,false
true,true,false,false,false,false
true,false,false,false,true,false
true,true,true,true,false,true
true,false,false,true,true,false
false,true,true,true,true,false
false,true,false,false,false,false
true,false,false,true,true,false"""

df = pd.read_csv(io.StringIO(csv_data))

def entropy(s):
    """Calculates Shannon entropy in bits."""
    probabilities = s.value_counts(normalize=True)
    return -np.sum(probabilities * np.log2(probabilities))

def conditional_entropy(df, feature, label):
    """Calculates H(Label | Feature)."""
    feature_probs = df[feature].value_counts(normalize=True)

    cond_entropy = 0
    for val, p_f in feature_probs.items():
        subset = df[df[feature] == val][label]
        cond_entropy += p_f * entropy(subset)
    return cond_entropy

def mutual_information(df, feature, label):
    """Calculates I(Feature; Label) = H(Label) - H(Label | Feature)."""
    return entropy(df[label]) - conditional_entropy(df, feature, label)

features = ['xA', 'xB', 'xC', 'xD', 'xE']
label = 'G'

# 1. Compute Entropies
entropies = {f: entropy(df[f]) for f in features}
h_label = entropy(df[label])

# 2. Compute Mutual Information (Information Gain)
mutual_infos = {f: mutual_information(df, f, label) for f in features}

print(f"--- Entropies (Bits) ---")
for f, h in entropies.items():
    print(f"H({f}): {h:.4f}")
print(f"H(Label {label}):  {h_label:.4f}\n")

print(f"--- Mutual Information (Information Gain) ---")
for f, mi in mutual_infos.items():
    print(f"I({f}; {label}): {mi:.4f}")

# Determine root node
best_feature = max(mutual_infos, key=mutual_infos.get)
print(f"\nRecommended Top Feature: {best_feature}")

--- Entropies (Bits) ---
H(xA): 0.9928
H(xB): 1.0000
H(xC): 0.9710
H(xD): 0.9928
H(xE): 0.9710
H(Label G):  0.8813

--- Mutual Information (Information Gain) ---
I(xA; G): 0.0173
I(xB; G): 0.0349
I(xC; G): 0.5568
I(xD; G): 0.0173
I(xE; G): 0.0058

Recommended Top Feature: xC
